In [ ]:
from __future__ import annotations

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                     TRATAMENTO DE DADOS FALTANTES                            ║
# ║           FIT Anti-Leakage, Imputação em Lote e Relatórios de Estratégia     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  IDENTIFICAÇÃO
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Autor       : Yago
#  Instituição : Universidade Federal do Ceará
#  Curso       : Engenharia Elétrica
#  Versão      : 3.3
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DESCRIÇÃO / FUNÇÃO DO SCRIPT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Este script implementa a etapa de tratamento de dados faltantes (Step 6) da
#  arquitetura de dados do TCC. Ele calcula estatísticas de imputação (FIT)
#  exclusivamente sobre o conjunto de treino e aplica essas estatísticas
#  (TRANSFORM) tanto ao treino quanto ao teste, sem nunca recalcular nada a
#  partir do teste — garantindo ausência de vazamento de informação (leakage)
#  entre os conjuntos.
#
#  O cenário configurado (FIT com todos os anos do treino) decide, para cada
#  variável numérica, uma estratégia de imputação com base no percentual de
#  valores ausentes: descarte, flag + mediana, interpolação linear ou
#  preenchimento temporal simples (ffill/bfill).
#
#  Ao final da execução, o script gera:
#    (a) Parquets imputados de treino e teste, por estratégia, dentro da
#        pasta do cenário;
#    (b) um JSON com as estatísticas de FIT (auditável, sem dados do teste);
#    (c) relatórios CSV de imputação por cenário e consolidado geral.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ENTRADAS (INPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  O script não requer entradas manuais do usuário. Os parâmetros de controle
#  estão definidos nas constantes abaixo (seção "CONFIGURAÇÕES GLOBAIS"):
#
#    SCENARIOS          : cenário(s) de FIT (padrão: todos os anos do treino,
#                          sem exclusão)
#    STRATEGIES         : estratégia(s) de imputação aplicadas no TRANSFORM
#                          (padrão: HYBRID_AGGRESSIVE)
#    THRESH_DESCARTE / THRESH_MISSFLAG / THRESH_SPLINE : limiares de % de NaN
#                          que definem a decisão de imputação por variável
#    COL_CHUNK_SIZE     : nº de colunas lidas por vez no FIT (otimização de
#                          I/O — ver seção de considerações técnicas)
#    DEFAULT_BATCH_SIZE : nº de linhas por lote no TRANSFORM
#    BASE_PATH          : caminho raiz de armazenamento
#                          (Google Drive: /content/drive/MyDrive/TCC_PLD_Project)
#
#  Arquivos de entrada esperados (gerados pela etapa anterior de divisão):
#    BASE_PATH / "...codigos/dados/5_divisão" / "treino" / "TREINO_SIN_BRASIL.parquet"
#    BASE_PATH / "...codigos/dados/5_divisão" / "teste"  / "TESTE_SIN_BRASIL.parquet"
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  SAÍDAS (OUTPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  1. Parquets imputados →  OUTPUT_BASE / <cenario> / {treino,teste} /
#     <ESTRATEGIA> / {TREINO,TESTE}_IMPUTADO.parquet   (compressão Snappy)
#
#  2. Estatísticas de FIT →  OUTPUT_BASE / <cenario> / "estatisticas_fit" /
#     fit_stats_treino.json
#     Contém, por variável: % de nulos, mediana, decisão de imputação e
#     metadados do FIT (linhas usadas, anos excluídos, limiares aplicados).
#
#  3. Relatórios (CSV)   →  OUTPUT_BASE / <cenario> / "relatorios" /
#     relatorio_imputacao.csv
#     OUTPUT_BASE / relatorio_consolidado_todos_cenarios.csv
#     Ambos com linhas, tempo de execução, tamanho de saída e nº de flags
#     geradas por split/estratégia/cenário.
#
#  4. Saída no terminal (Rich):
#     • Painéis de FIT e TRANSFORM por cenário/split/estratégia
#     • Tabela de distribuição de variáveis por faixa de % de NaN
#     • Tabela consolidada final com status OK/ERRO por execução
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ARQUITETURA ANTI-LEAKAGE
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  FIT (todos os anos do treino) → TRANSFORM aplicado ao treino e ao teste.
#  O TESTE nunca influencia nenhum parâmetro de fit (medianas, percentuais de
#  nulos e decisões de imputação vêm exclusivamente do treino).
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CONSIDERAÇÕES INICIAIS E OBSERVAÇÕES TÉCNICAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
#  1. AMBIENTE DE EXECUÇÃO
#     O script foi desenvolvido e testado no Google Colab (Python 3.10+).
#     Pode ser executado localmente, mas requer ajuste manual de BASE_PATH.
#     Depende dos Parquets de treino/teste gerados pela etapa anterior de
#     divisão (Step 5).
#
#  2. OTIMIZAÇÕES DE LEITURA NO FIT (vs. versão anterior por coluna)
#     Antes: uma leitura por coluna (pq.read_table(columns=[col])) — do
#     tamanho do catálogo de variáveis, o que gerava milhares de leituras
#     no Drive.
#     Agora: leitura em blocos de COL_CHUNK_SIZE colunas por vez, reduzindo
#     drasticamente o número de leituras, com mediana calculada de forma
#     vetorizada (np.nanmedian) sobre o bloco inteiro e o filtro de anos
#     aplicado via máscara numpy — sem laços Python linha a linha.
#
#  3. OTIMIZAÇÕES NO TRANSFORM
#     Antes: pd.concat dentro do loop de colunas (uma realocação por
#     coluna/lote) e cast float64→float32 coluna a coluna.
#     Agora: as flags de imputação são acumuladas em arrays numpy e
#     concatenadas em UM único pd.concat por lote; o cast de tipos é feito
#     em bloco sobre todas as colunas float64 de uma vez; a ordenação por
#     chave temporal (sort_values) ocorre apenas no primeiro lote.
#
#  4. CONSUMO DE MEMÓRIA POR CHUNK (FIT)
#     COL_CHUNK_SIZE colunas × linhas do treino × float32 — por exemplo,
#     500 colunas × ~43 mil linhas × 4 bytes ≈ 86 MB por chunk, descartado
#     da memória logo após o cálculo da mediana. Caso ocorra estouro de
#     memória (OOM), reduzir COL_CHUNK_SIZE (ex.: para 200) resolve.
#
#  5. DECISÃO DE IMPUTAÇÃO POR VARIÁVEL
#     Cada variável numérica recebe uma decisão com base no % de nulos no
#     treino, conforme os limiares THRESH_DESCARTE / THRESH_MISSFLAG /
#     THRESH_SPLINE:
#       >= 50%  → DISCARD (variável zerada, não descartada fisicamente)
#       > 15%   → MISSFLAG_MEDIAN (flag binária + preenchimento por mediana)
#       > 5%    → LINEAR (interpolação linear)
#       <= 5%   → FFILL_BFILL (preenchimento temporal simples)
#     Essas decisões são calculadas uma única vez no FIT e reaplicadas
#     identicamente ao treino e ao teste no TRANSFORM.
#
#  6. ESTRATÉGIAS DE IMPUTAÇÃO DISPONÍVEIS
#     HYBRID_AGGRESSIVE (padrão/benchmark do TCC), ZERO_MISSFLAG (baseline
#     conservador), LINEAR_FULL (indicado para séries meteorológicas
#     contínuas) e TIME_ONLY (referência mínima). Cada uma tem uma função
#     `_apply()` própria dentro de ImputationTransformer.
#
#  7. CONTINUIDADE ENTRE LOTES (TRANSFORM)
#     O TRANSFORM processa o Parquet em lotes (batches) via
#     iter_batches(). Para evitar descontinuidade nas bordas de lote,
#     o último valor válido de cada coluna é guardado (`persistence`) e
#     usado para preencher o primeiro valor ausente do lote seguinte antes
#     de aplicar a estratégia de imputação.
#
#  8. RETRY DE LEITURA
#     _open_parquet_with_retry() tenta reabrir o Parquet de entrada até
#     3 vezes com espera entre tentativas, mitigando falhas transitórias
#     de acesso ao Google Drive.
#
#  9. ESCRITA EM DISCO LOCAL COM CÓPIA FINAL
#     Os Parquets de saída são escritos primeiro em DIR_TMP (disco local,
#     mais rápido) e só depois copiados para o destino final no Drive,
#     reduzindo o tempo de escrita incremental via ParquetWriter.
#
#  10. REPRODUTIBILIDADE
#      O JSON de estatísticas de FIT inclui metadados completos (linhas
#      usadas, anos excluídos, limiares aplicados, tamanho de chunk),
#      permitindo recarregar o FIT sem reprocessamento via
#      `reload_stats=True` e `ImputationFitter.from_json()`.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DEPENDÊNCIAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Biblioteca        Versão mínima  Finalidade
#  numpy             1.23           Operações vetorizadas, mediana, máscaras
#  pandas            1.5            Manipulação de DataFrames e imputação
#  pyarrow           10.0           Leitura/escrita Parquet em lotes (batches)
#  rich              13.0           Painéis, tabelas e progresso no terminal
#
#  Instalação rápida (Google Colab / pip):
#      !pip install rich pyarrow
#  (numpy e pandas já estão disponíveis no Colab por padrão)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ESTRUTURA DE DIRETÓRIOS GERADA
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  .../6_tratamento_de_faltantes/
#  ├── cenario_A_todos_anos/
#  │   ├── treino/<ESTRATEGIA>/TREINO_IMPUTADO.parquet
#  │   ├── teste/<ESTRATEGIA>/TESTE_IMPUTADO.parquet
#  │   ├── estatisticas_fit/fit_stats_treino.json
#  │   └── relatorios/relatorio_imputacao.csv
#  └── relatorio_consolidado_todos_cenarios.csv
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  COMO EXECUTAR (Google Colab)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Célula 1 — Montar o Drive:
#      from google.colab import drive
#      drive.mount('/content/drive')
#
#  Célula 2 — Instalar dependências:
#      !pip install rich pyarrow
#
#  Célula 3 — Executar o script:
#      exec(open('tratamento_faltantes_tcc_v33.py').read())
#   ou simplesmente executar este arquivo como módulo principal.
#
# ══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")

import gc
import json
import shutil
import time
from contextlib import contextmanager
from pathlib import Path
from typing import Dict, Generator, List, Optional

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from rich.console import Console
from rich.panel import Panel
from rich.progress import (
    BarColumn, MofNCompleteColumn, Progress,
    SpinnerColumn, TextColumn, TimeElapsedColumn,
)
from rich.table import Table
from rich.theme import Theme
from rich import box

# ==============================================================================
# 🎨 TEMA E CONSOLE GLOBAL
# ==============================================================================

_THEME = Theme({
    "info":      "bold cyan",
    "warning":   "bold yellow",
    "error":     "bold red",
    "success":   "bold green",
    "header":    "bold white on blue",
    "metric":    "bold magenta",
    "dim":       "dim white",
    "treino":    "bold steel_blue1",
    "teste":     "bold orange1",
    "cenario_a": "bold green",
})

console = Console(theme=_THEME)


# ==============================================================================
# ⚙️ CONFIGURAÇÕES GLOBAIS
# ==============================================================================

BASE_PATH = Path("/content/drive/MyDrive/TCC_PLD_Project")

DIR_SPLIT    = BASE_PATH / "09-ESCRITA_TCC/PARTES_TCC/codigos/dados/5_divisão"
INPUT_TREINO = DIR_SPLIT / "treino" / "TREINO_SIN_BRASIL.parquet"
INPUT_TESTE  = DIR_SPLIT / "teste"  / "TESTE_SIN_BRASIL.parquet"

OUTPUT_BASE = BASE_PATH / "09-ESCRITA_TCC/PARTES_TCC/codigos/dados/6_tratamento_de_faltantes"

DIR_TMP = Path("/tmp/missing_engine_v33")

TIME_COLS: List[str] = ["KEY_ANO", "KEY_MES", "KEY_DIA", "KEY_HORA"]

SCENARIOS: List[Dict] = [
    {
        "label":         "cenario_A_todos_anos",
        "exclude_years": [],
        "desc":          "FIT com todos os anos do treino",
    },
]

THRESH_DESCARTE = 0.50
THRESH_MISSFLAG = 0.15
THRESH_SPLINE   = 0.05

# ── Leitura em blocos de COL_CHUNK_SIZE colunas por vez no FIT ───────────────
# 500 colunas × 43k linhas × float32 ≈ 86 MB por chunk.
# Se der OOM, reduza para 200 (≈ 34 MB).
COL_CHUNK_SIZE     = 500
DEFAULT_BATCH_SIZE = 15_000

STRATEGIES: List[str] = ["HYBRID_AGGRESSIVE"]

STRATEGY_META: Dict[str, Dict[str, str]] = {
    "HYBRID_AGGRESSIVE": {
        "desc":  f"Descarte(>={THRESH_DESCARTE*100:.0f}%) + Flag+Mediana(>{THRESH_MISSFLAG*100:.0f}%) + Linear(>{THRESH_SPLINE*100:.0f}%) + ffill/bfill",
        "uso":   "Benchmark principal do TCC",
        "risco": "Baixo",
    },
    "ZERO_MISSFLAG": {
        "desc":  "Flag binária + preenchimento por zero",
        "uso":   "Baseline conservador / tree-based models",
        "risco": "Médio (distorce média)",
    },
    "LINEAR_FULL": {
        "desc":  "Interpolação linear + ffill/bfill completo",
        "uso":   "Séries temporais contínuas (METEO)",
        "risco": "Baixo (preserva tendência)",
    },
    "TIME_ONLY": {
        "desc":  "Apenas ffill/bfill temporal simples",
        "uso":   "Referência mínima / comparação",
        "risco": "Alto (propaga último valor)",
    },
}

_NUMERIC_PA_TYPES = frozenset([
    pa.int8(), pa.int16(), pa.int32(), pa.int64(),
    pa.uint8(), pa.uint16(), pa.uint32(), pa.uint64(),
    pa.float16(), pa.float32(), pa.float64(),
])


# ==============================================================================
# ⏱️ UTILITÁRIOS
# ==============================================================================

@contextmanager
def timer(label: str) -> Generator:
    t0 = time.perf_counter()
    yield
    console.print(f"   [dim]⏱  {label}: [/][metric]{time.perf_counter()-t0:.2f}s[/]")


def fmt_n(n: int) -> str:
    return f"{n:,}".replace(",", ".")


def bytes_to_mb(b: int) -> float:
    return round(b / 1024 ** 2, 2)


def _open_parquet_with_retry(path: Path, retries: int = 3, delay: float = 5.0) -> pq.ParquetFile:
    last_exc: Exception = RuntimeError("Nenhuma tentativa realizada")
    for attempt in range(1, retries + 1):
        try:
            return pq.ParquetFile(str(path))
        except Exception as exc:
            last_exc = exc
            console.print(
                f"   [warning]⚠  Tentativa {attempt}/{retries} → {path.name}: {exc}. "
                f"Aguardando {delay}s...[/]"
            )
            time.sleep(delay)
    raise last_exc


def _safe_cast_table(table: pa.Table, schema: pa.Schema) -> pa.Table:
    arrays = []
    for field in schema:
        if field.name in table.schema.names:
            col = table.column(field.name)
            try:
                arrays.append(col.cast(field.type))
            except (pa.ArrowInvalid, pa.ArrowNotImplementedError):
                arrays.append(col)
        else:
            arrays.append(pa.array([None] * len(table), type=field.type))
    return pa.table(arrays, schema=schema)


def _get_numeric_cols(df: pd.DataFrame) -> List[str]:
    num = df.select_dtypes(include=[np.number]).columns.tolist()
    return [c for c in num if c not in TIME_COLS]


# ==============================================================================
# 📐 FITTER v3.3-OPT — chunk de colunas + mediana vetorizada + filtro de anos
# ==============================================================================

class ImputationFitter:
    """
    FIT-ONLY com duas otimizações principais vs versão original:

    1. Leitura em chunks de COL_CHUNK_SIZE colunas por vez:
       Reduz drasticamente o número de leituras no Drive em comparação com
       ler uma coluna por vez.

    2. Mediana vetorizada com np.nanmedian:
       Opera sobre o bloco inteiro (linhas × cols_chunk) em C,
       em vez de um loop Python por coluna.

    3. Filtro de anos via máscara numpy:
       A máscara booleana é construída UMA VEZ lendo só KEY_ANO,
       depois aplicada com arr[mask] — sem loop Python por linha.

    RAM por chunk: COL_CHUNK_SIZE × fit_rows × float32
                   500 × 43k × 4 bytes ≈ 86 MB (descartado após mediana).
    """

    def __init__(
        self,
        treino_path:   Path,
        exclude_years: List[int] = None,
        col_chunk:     int       = COL_CHUNK_SIZE,
    ) -> None:
        self.treino_path   = treino_path
        self.exclude_years = exclude_years or []
        self.col_chunk     = col_chunk
        self.stats: Dict[str, Dict] = {}
        self._total_rows = 0
        self._fit_rows   = 0

    def _build_year_mask(self) -> np.ndarray:
        """
        Retorna array booleano (total_rows,): True = linha incluída no fit.
        Lê apenas a coluna KEY_ANO — custo único, ~350 KB para 43k linhas.
        """
        if not self.exclude_years:
            mask = np.ones(self._total_rows, dtype=bool)
            console.print(
                f"   [dim]Filtro de anos: nenhum excluído — "
                f"todas as {fmt_n(self._total_rows)} linhas usadas.[/]\n"
            )
            return mask

        schema_names = pq.read_schema(str(self.treino_path)).names
        if "KEY_ANO" not in schema_names:
            console.print("   [warning]⚠  KEY_ANO não encontrada — nenhuma linha excluída.[/]")
            return np.ones(self._total_rows, dtype=bool)

        # Lê KEY_ANO como array numpy (uma única leitura)
        anos = (
            pq.read_table(str(self.treino_path), columns=["KEY_ANO"])
            .column("KEY_ANO")
            .to_pylist()
        )
        excl_set = set(self.exclude_years)
        mask     = np.array([a not in excl_set for a in anos], dtype=bool)
        n_excl   = int((~mask).sum())
        n_incl   = int(mask.sum())
        console.print(
            f"   [info]Filtro de anos:[/] excluindo [warning]{self.exclude_years}[/] → "
            f"[metric]{fmt_n(n_excl)}[/] linhas removidas, "
            f"[metric]{fmt_n(n_incl)}[/] utilizadas.\n"
        )
        return mask

    def fit(self, scenario_label: str = "") -> "ImputationFitter":
        if not self.treino_path.exists():
            raise FileNotFoundError(f"Treino não encontrado: {self.treino_path}")

        excl_str = (
            f"[warning]Anos excluídos: {self.exclude_years}[/]"
            if self.exclude_years
            else "[dim]Sem exclusão de anos (todos utilizados)[/]"
        )
        console.print(
            Panel(
                f"[treino]FIT — estatísticas calculadas APENAS no TREINO[/]\n"
                f"[dim]Medianas aplicadas ao TESTE sem recálculo (anti-leakage)[/]\n"
                f"[dim]Modo v3.3-OPT: chunks de {self.col_chunk} colunas + mediana vetorizada[/]\n"
                f"{excl_str}",
                title=f"[header] FIT — {scenario_label or 'TREINO'} [/]",
                border_style="steel_blue1",
                padding=(0, 2),
            )
        )

        # Fase 1: schema + linhas
        console.print("   [dim]Fase 1/3 — schema e contagem de linhas...[/]")
        schema           = pq.read_schema(str(self.treino_path))
        pf               = _open_parquet_with_retry(self.treino_path)
        self._total_rows = pf.metadata.num_rows

        numeric_cols: List[str] = [
            c for c in schema.names
            if c not in TIME_COLS
            and schema.field(c).type in _NUMERIC_PA_TYPES
        ]
        n_cols   = len(numeric_cols)
        n_chunks = (n_cols + self.col_chunk - 1) // self.col_chunk

        console.print(
            f"   [info]Linhas totais no treino[/]: [metric]{fmt_n(self._total_rows)}[/]  |  "
            f"[info]Colunas numéricas[/]: [metric]{fmt_n(n_cols)}[/]\n"
            f"   [dim]Leituras Drive: {n_chunks} chunks de {self.col_chunk} colunas[/]\n"
        )

        # Fase 2: máscara de anos (uma leitura de KEY_ANO)
        console.print("   [dim]Fase 2/3 — máscara de anos...[/]")
        include_mask   = self._build_year_mask()
        self._fit_rows = int(include_mask.sum())

        # Fase 3: chunks de colunas + mediana vetorizada
        console.print("   [dim]Fase 3/3 — NaN% + mediana por chunk de colunas...[/]")

        with Progress(
            SpinnerColumn(),
            TextColumn("[treino]{task.description}[/]"),
            BarColumn(),
            TextColumn("[metric]{task.completed}/{task.total}[/]"),
            TimeElapsedColumn(),
            console=console,
            transient=True,
        ) as progress:
            task = progress.add_task("FIT — chunks...", total=n_chunks)

            for chunk_start in range(0, n_cols, self.col_chunk):
                chunk_cols = numeric_cols[chunk_start: chunk_start + self.col_chunk]

                try:
                    # Uma leitura para COL_CHUNK_SIZE colunas inteiras
                    arr_full = (
                        pq.read_table(str(self.treino_path), columns=chunk_cols)
                        .to_pandas()
                        .values
                        .astype(np.float32, copy=False)
                    )  # shape: (total_rows, len(chunk_cols))

                    # Aplica a máscara de anos via indexação numpy (vetorizado)
                    arr = arr_full[include_mask]   # shape: (fit_rows, len(chunk_cols))
                    del arr_full
                    gc.collect()

                    # Mediana e contagem de NaN — ambos vetorizados sobre o eixo 0
                    n_nulls_vec = np.isnan(arr).sum(axis=0)   # (len(chunk_cols),)
                    medians_vec = np.nanmedian(arr, axis=0)   # (len(chunk_cols),)
                    del arr
                    gc.collect()

                except Exception as exc:
                    console.print(f"   [warning]⚠  Chunk {chunk_start}: {exc} — usando zero[/]")
                    n_nulls_vec = np.zeros(len(chunk_cols), dtype=np.int64)
                    medians_vec = np.zeros(len(chunk_cols), dtype=np.float32)

                # Popula stats (só dict Python — rápido)
                for i, col in enumerate(chunk_cols):
                    n_nulls = int(n_nulls_vec[i])
                    mediana = float(medians_vec[i]) if not np.isnan(medians_vec[i]) else 0.0
                    pct     = n_nulls / self._fit_rows if self._fit_rows > 0 else 0.0

                    if pct >= THRESH_DESCARTE:
                        decisao = "DISCARD"
                    elif pct > THRESH_MISSFLAG:
                        decisao = "MISSFLAG_MEDIAN"
                    elif pct > THRESH_SPLINE:
                        decisao = "LINEAR"
                    else:
                        decisao = "FFILL_BFILL"

                    self.stats[col] = {
                        "null_pct":  round(pct, 6),
                        "has_nulls": n_nulls > 0,
                        "n_nulls":   n_nulls,
                        "mediana":   mediana,
                        "decisao":   decisao,
                    }

                progress.advance(task)

        del include_mask
        gc.collect()

        self._print_fit_summary(scenario_label)
        return self

    def _print_fit_summary(self, scenario_label: str = "") -> None:
        bins = {
            "DISCARD (>=50%)":        sum(1 for v in self.stats.values() if v["decisao"] == "DISCARD"),
            "MISSFLAG_MEDIAN (>15%)": sum(1 for v in self.stats.values() if v["decisao"] == "MISSFLAG_MEDIAN"),
            "LINEAR (>5%)":           sum(1 for v in self.stats.values() if v["decisao"] == "LINEAR"),
            "FFILL_BFILL (<=5%)":     sum(1 for v in self.stats.values() if v["decisao"] == "FFILL_BFILL"),
            "Sem NaN":                sum(1 for v in self.stats.values() if not v["has_nulls"]),
        }
        total    = len(self.stats)
        excl_info = f"excluindo {self.exclude_years}" if self.exclude_years else "todos os anos"

        t = Table(
            title=(
                f"[{scenario_label}] Distribuição de Variáveis por Faixa de NaN "
                f"({fmt_n(self._fit_rows)} linhas — {excl_info})"
            ),
            border_style="steel_blue1", box=box.SIMPLE_HEAD, show_lines=True,
        )
        t.add_column("Decisão de Imputação", style="bold white", justify="left",  min_width=28)
        t.add_column("Qtd. Variáveis",       style="metric",     justify="right", min_width=14)
        t.add_column("% do Total",           style="dim",        justify="right", min_width=10)

        color_map = {
            "DISCARD (>=50%)": "red", "MISSFLAG_MEDIAN (>15%)": "yellow",
            "LINEAR (>5%)": "cyan",   "FFILL_BFILL (<=5%)": "green", "Sem NaN": "bright_green",
        }
        for label, qtd in bins.items():
            pct_str = f"{qtd / total * 100:.1f}%" if total else "—"
            cor     = color_map.get(label, "white")
            t.add_row(f"[{cor}]{label}[/{cor}]", f"{qtd:,}", pct_str)

        console.print(t)
        console.print(
            f"   [success]✔  FIT concluído ({scenario_label}):[/] "
            f"{fmt_n(total)} variáveis | [metric]{fmt_n(self._fit_rows)}[/] linhas usadas\n"
        )

    def save_stats(self, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "__meta__": {
                "scenario":          "cenario_A",
                "fit_rows":          self._fit_rows,
                "total_rows_treino": self._total_rows,
                "excluded_years":    self.exclude_years,
                "thresh_descarte":   THRESH_DESCARTE,
                "thresh_missflag":   THRESH_MISSFLAG,
                "thresh_spline":     THRESH_SPLINE,
                "col_chunk":         self.col_chunk,
            },
            **self.stats,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)
        console.print(f"   [success]✔  Stats salvas:[/] [dim]{path}[/]\n")

    @classmethod
    def from_json(cls, path: Path) -> "ImputationFitter":
        obj = cls.__new__(cls)
        obj.col_chunk     = COL_CHUNK_SIZE
        obj._total_rows   = 0
        obj._fit_rows     = 0
        with open(path, encoding="utf-8") as f:
            raw = json.load(f)
        meta              = raw.pop("__meta__", {})
        obj._fit_rows     = meta.get("fit_rows", 0)
        obj._total_rows   = meta.get("total_rows_treino", 0)
        obj.exclude_years = meta.get("excluded_years", [])
        obj.treino_path   = Path("")
        obj.stats         = raw
        console.print(
            f"   [info]Stats carregadas:[/] [dim]{path}[/] "
            f"[metric]({len(obj.stats):,} variáveis)[/] "
            f"[dim]| excluídos: {obj.exclude_years}[/]\n"
        )
        return obj


# ==============================================================================
# 🔧 TRANSFORMER v3.3-OPT — sem pd.concat no loop + cast vetorizado
# ==============================================================================

class ImputationTransformer:
    """
    TRANSFORM-ONLY: aplica estatísticas do FIT sem recalcular — zero data leakage.

    Otimizações vs versão original:
      • flags acumuladas como arrays numpy → um único pd.concat POR BATCH
      • cast float64→float32 em bloco: df[float64_cols].astype("float32")
      • sort_values apenas no primeiro batch
    """

    def __init__(
        self,
        stats:      Dict[str, Dict],
        strategy:   str,
        batch_size: int = DEFAULT_BATCH_SIZE,
    ) -> None:
        self.stats      = stats
        self.strategy   = strategy
        self.batch_size = batch_size

    def transform(
        self,
        input_path:        Path,
        output_path_tmp:   Path,
        output_path_final: Path,
        split_label:       str = "TREINO",
    ) -> Dict:
        if not input_path.exists():
            raise FileNotFoundError(f"Input não encontrado: {input_path}")

        cor  = "treino" if split_label == "TREINO" else "teste"
        meta = STRATEGY_META.get(self.strategy, {})

        console.print(
            Panel(
                f"[{cor}]Split: {split_label}[/]  |  [info]Estratégia: {self.strategy}[/]\n"
                f"[dim]{meta.get('desc', '')}[/]\n"
                f"[dim]Uso: {meta.get('uso', '')} | Risco: {meta.get('risco', '')}[/]",
                title=f"[header] TRANSFORM — {split_label} [/]",
                border_style="cyan" if split_label == "TREINO" else "orange1",
                padding=(0, 2),
            )
        )

        p_file           = _open_parquet_with_retry(input_path)
        persistence:     Dict[str, float]           = {}
        writer:          Optional[pq.ParquetWriter] = None
        expected_schema: Optional[pa.Schema]        = None
        expected_cols:   Optional[List[str]]        = None

        t0          = time.perf_counter()
        total_rows  = 0
        first_batch = True

        output_path_tmp.parent.mkdir(parents=True, exist_ok=True)

        try:
            with Progress(
                SpinnerColumn(),
                TextColumn(f"[{cor}]{{task.description}}[/]"),
                BarColumn(), MofNCompleteColumn(), TimeElapsedColumn(),
                console=console, transient=True,
            ) as progress:
                task = progress.add_task(
                    f"Imputando {split_label} — {self.strategy}...", total=None
                )

                for batch in p_file.iter_batches(batch_size=self.batch_size):
                    df = batch.to_pandas()
                    total_rows += len(df)

                    # Sort apenas no primeiro batch
                    if first_batch and all(c in df.columns for c in TIME_COLS):
                        df = df.sort_values(TIME_COLS).reset_index(drop=True)
                        first_batch = False

                    num_cols  = _get_numeric_cols(df)
                    new_flags: Dict[str, np.ndarray] = {}  # arrays, não Series

                    for col in num_cols:
                        col_stat = self.stats.get(col, {})
                        decisao  = col_stat.get("decisao", "FFILL_BFILL")
                        mediana  = float(col_stat.get("mediana", 0.0))

                        # Continuidade entre batches
                        last_val = persistence.get(col)
                        if last_val is not None and pd.isna(df.iloc[0][col]):
                            df.at[df.index[0], col] = last_val

                        df = self._apply(df, col, decisao, mediana, new_flags)

                        if df[col].isna().any():
                            df[col] = df[col].fillna(mediana if pd.notna(mediana) else 0.0)

                        persistence[col] = float(df[col].iloc[-1])

                    # Cast float64→float32 em bloco (vetorizado)
                    float64_cols = [
                        c for c in df.select_dtypes(include=["float64"]).columns
                        if c not in TIME_COLS
                    ]
                    if float64_cols:
                        df[float64_cols] = df[float64_cols].astype("float32")

                    # Um único pd.concat por batch (não por coluna)
                    if new_flags:
                        flags_df = pd.DataFrame(new_flags, index=df.index)
                        df = pd.concat([df, flags_df], axis=1)
                        del flags_df

                    if writer is None:
                        expected_cols   = df.columns.tolist()
                        table_out       = pa.Table.from_pandas(df, preserve_index=False)
                        expected_schema = table_out.schema
                        writer = pq.ParquetWriter(
                            str(output_path_tmp), expected_schema, compression="snappy"
                        )
                    else:
                        for col in expected_cols:  # type: ignore[union-attr]
                            if col not in df.columns:
                                df[col] = np.int8(0) if col.startswith("FLAG_") else np.float32(0.0)
                        df        = df[expected_cols]
                        table_out = pa.Table.from_pandas(df, preserve_index=False)
                        table_out = _safe_cast_table(table_out, expected_schema)  # type: ignore[arg-type]

                    writer.write_table(table_out)

                    del df, table_out
                    gc.collect()
                    progress.advance(task)

        finally:
            if writer is not None:
                try:
                    writer.close()
                except Exception as e:
                    console.print(f"   [warning]⚠  Erro ao fechar writer: {e}[/]")

        output_path_final.parent.mkdir(parents=True, exist_ok=True)
        console.print(f"   [dim]Copiando {output_path_tmp.name} → Drive...[/]")
        shutil.copy2(str(output_path_tmp), str(output_path_final))
        output_path_tmp.unlink(missing_ok=True)

        elapsed     = time.perf_counter() - t0
        out_size_mb = bytes_to_mb(output_path_final.stat().st_size)
        n_flags_uni = len({c for c in (expected_cols or []) if c.startswith("FLAG_")})

        result = {
            "split":     split_label,
            "strategy":  self.strategy,
            "rows":      total_rows,
            "elapsed_s": round(elapsed, 2),
            "output_mb": out_size_mb,
            "n_flags":   n_flags_uni,
            "path":      str(output_path_final),
            "status":    "OK",
        }

        console.print(
            f"   [success]✔ {split_label} — {self.strategy}[/] "
            f"| {fmt_n(total_rows)} linhas "
            f"| [metric]{elapsed:.2f}s[/] "
            f"| [metric]{out_size_mb} MB[/] "
            f"| Flags: [metric]{n_flags_uni:,}[/]\n"
        )
        return result

    def _apply(
        self,
        df:        pd.DataFrame,
        col:       str,
        decisao:   str,
        mediana:   float,
        new_flags: Dict[str, np.ndarray],
    ) -> pd.DataFrame:
        if self.strategy == "HYBRID_AGGRESSIVE":
            if decisao == "DISCARD":
                df[col] = np.float32(0.0)
            elif decisao == "MISSFLAG_MEDIAN":
                mask = df[col].isna()
                if mask.any():
                    new_flags[f"FLAG_{col}"] = mask.to_numpy(dtype=np.int8)
                    df[col] = df[col].fillna(mediana)
            elif decisao == "LINEAR":
                df[col] = df[col].interpolate(method="linear", limit_direction="both")
            df[col] = df[col].ffill().bfill()

        elif self.strategy == "ZERO_MISSFLAG":
            mask = df[col].isna()
            if mask.any():
                new_flags[f"FLAG_{col}"] = mask.to_numpy(dtype=np.int8)
                df[col] = df[col].fillna(np.float32(0.0))

        elif self.strategy == "LINEAR_FULL":
            df[col] = (
                df[col]
                .interpolate(method="linear", limit_direction="both")
                .ffill().bfill()
            )

        else:  # TIME_ONLY
            df[col] = df[col].ffill().bfill()

        return df


# ==============================================================================
# 🚀 MOTOR PRINCIPAL — Cenário único (todos os anos)
# ==============================================================================

class MissingEngineV33:
    """
    Para o cenário configurado em SCENARIOS:
      1. Faz o FIT com as linhas correspondentes (chunk de colunas + numpy vetorizado)
      2. Aplica TRANSFORM no treino e teste para cada estratégia
      3. Salva outputs em subpastas separadas por cenário
    """

    def __init__(
        self,
        input_treino:  Path                 = INPUT_TREINO,
        input_teste:   Path                 = INPUT_TESTE,
        output_base:   Path                 = OUTPUT_BASE,
        batch_size:    int                  = DEFAULT_BATCH_SIZE,
        col_chunk:     int                  = COL_CHUNK_SIZE,
        strategies:    Optional[List[str]]  = None,
        scenarios:     Optional[List[Dict]] = None,
        reload_stats:  bool                 = False,
    ) -> None:
        self.input_treino = Path(input_treino)
        self.input_teste  = Path(input_teste)
        self.output_base  = Path(output_base)
        self.batch_size   = batch_size
        self.col_chunk    = col_chunk
        self.strategies   = strategies or STRATEGIES
        self.scenarios    = scenarios  or SCENARIOS
        self.reload_stats = reload_stats
        self._all_results: List[Dict] = []

        DIR_TMP.mkdir(parents=True, exist_ok=True)
        self.output_base.mkdir(parents=True, exist_ok=True)

    def run(self) -> None:
        self._print_banner()
        t_total = time.perf_counter()

        for scenario in self.scenarios:
            self._run_scenario(scenario)

        elapsed_total = time.perf_counter() - t_total
        self._print_final_report(elapsed_total)
        self._salvar_relatorio_consolidado()

    def _run_scenario(self, scenario: Dict) -> None:
        label         = scenario["label"]
        exclude_years = scenario["exclude_years"]
        desc          = scenario["desc"]

        cor    = "cenario_a"
        titulo = "CENÁRIO A — TODOS OS ANOS"

        console.print()
        console.rule(f"[{cor}]{'=' * 20}  {titulo}  {'=' * 20}[/]")
        console.print(
            Panel(
                f"[{cor}]{desc}[/]\n"
                f"[dim]Pasta de saída: {self.output_base / label}[/]\n"
                f"[dim]Estratégias: {', '.join(self.strategies)}[/]",
                title=f"[header] {titulo} [/]",
                border_style="green",
                padding=(0, 2),
            )
        )

        dir_treino_out = self.output_base / label / "treino"
        dir_teste_out  = self.output_base / label / "teste"
        dir_stats_out  = self.output_base / label / "estatisticas_fit"
        dir_relat_out  = self.output_base / label / "relatorios"
        stats_path     = dir_stats_out / "fit_stats_treino.json"

        # ── FIT ───────────────────────────────────────────────────────────────
        if self.reload_stats and stats_path.exists():
            console.print(
                Panel(
                    f"[info]Reutilizando stats:[/] [dim]{stats_path}[/]",
                    title="[header] FIT (RELOAD) [/]",
                    border_style="steel_blue1",
                )
            )
            fitter = ImputationFitter.from_json(stats_path)
        else:
            with timer(f"FIT — {label}"):
                fitter = ImputationFitter(
                    treino_path   = self.input_treino,
                    exclude_years = exclude_years,
                    col_chunk     = self.col_chunk,
                ).fit(scenario_label=label)
            fitter.save_stats(stats_path)

        # ── TRANSFORM por estratégia ──────────────────────────────────────────
        scenario_results: List[Dict] = []

        for strategy in self.strategies:
            console.rule(f"[header] {label} | ESTRATÉGIA: {strategy} [/]")
            transformer = ImputationTransformer(
                stats      = fitter.stats,
                strategy   = strategy,
                batch_size = self.batch_size,
            )

            for split_label, input_path, out_sub in [
                ("TREINO", self.input_treino, dir_treino_out),
                ("TESTE",  self.input_teste,  dir_teste_out),
            ]:
                try:
                    r = transformer.transform(
                        input_path        = input_path,
                        output_path_tmp   = DIR_TMP / f"{label}_{split_label}_{strategy}.parquet",
                        output_path_final = out_sub / strategy / f"{split_label}_IMPUTADO.parquet",
                        split_label       = split_label,
                    )
                    r["cenario"] = label
                    scenario_results.append(r)
                except Exception:
                    console.print_exception()
                    scenario_results.append(self._erro(split_label, strategy, label))

        self._all_results.extend(scenario_results)
        self._salvar_relatorio_cenario(scenario_results, dir_relat_out)

    @staticmethod
    def _erro(split: str, strategy: str, cenario: str) -> Dict:
        return {
            "cenario": cenario, "split": split, "strategy": strategy,
            "rows": 0, "elapsed_s": 0, "output_mb": 0,
            "n_flags": 0, "path": "—", "status": "ERRO",
        }

    def _print_banner(self) -> None:
        cenarios_str = "\n".join(
            f"  [cenario_a]A[/cenario_a]"
            f"  [dim]{s['label']:35s} | {s['desc']}[/dim]"
            for s in self.scenarios
        )
        console.print()
        console.print(
            Panel(
                "[bold white]MOTOR DE TRATAMENTO DE FALTANTES v3.3-OPT — CENÁRIO ÚNICO + CHUNK FIT[/]\n"
                "[dim]FIT em chunks de colunas (PyArrow) | TRANSFORM em batches | Anti-Leakage[/]\n\n"
                f"[dim]Input TREINO  : {self.input_treino}[/]\n"
                f"[dim]Input TESTE   : {self.input_teste}[/]\n"
                f"[dim]Output base   : {self.output_base}[/]\n"
                f"[dim]Col chunk     : {self.col_chunk} colunas/leitura[/]\n\n"
                f"[bold white]Cenário:[/]\n{cenarios_str}\n\n"
                f"[dim]Estratégias   : {', '.join(self.strategies)}[/]",
                title="[header]  TCC PLD HORÁRIO — STEP 6: TRATAMENTO DE FALTANTES  [/]",
                border_style="blue",
                padding=(1, 4),
            )
        )
        console.print(
            Panel(
                "[warning]GARANTIA ANTI-LEAKAGE:[/]\n\n"
                "  [cenario_a]Cenário A[/]  →  FIT com todos os anos  →  TRANSFORM treino + teste\n\n"
                "  [dim]O TESTE nunca influencia nenhum parâmetro de fit.[/]\n"
                "  [dim]Stats auditáveis em: <cenario>/estatisticas_fit/fit_stats_treino.json[/]",
                title="[header] ARQUITETURA ANTI-LEAKAGE [/]",
                border_style="yellow",
                padding=(0, 2),
            )
        )

    def _print_final_report(self, elapsed_total: float) -> None:
        t = Table(
            title="RELATÓRIO CONSOLIDADO — CENÁRIO E ESTRATÉGIAS",
            border_style="blue", box=box.DOUBLE_EDGE, show_lines=True, min_width=140,
        )
        t.add_column("Cenário",    style="bold",      justify="left",   min_width=28)
        t.add_column("Split",      style="bold",      justify="center", min_width=8)
        t.add_column("Estratégia", style="bold cyan",  justify="left",   min_width=22)
        t.add_column("Registros",  style="metric",    justify="right",  min_width=12)
        t.add_column("Flags",      style="yellow",    justify="right",  min_width=8)
        t.add_column("Saída (MB)", style="green",     justify="right",  min_width=10)
        t.add_column("Tempo (s)",  style="white",     justify="right",  min_width=9)
        t.add_column("Status",     style="bold",      justify="center", min_width=7)

        for r in self._all_results:
            cor_c  = "cenario_a"
            cor_s  = "treino" if r["split"] == "TREINO" else "teste"
            status = "[green]OK[/]" if r["status"] == "OK" else "[red]ERRO[/]"
            t.add_row(
                f"[{cor_c}]{r.get('cenario', '—')}[/{cor_c}]",
                f"[{cor_s}]{r['split']}[/{cor_s}]",
                r["strategy"],
                fmt_n(r["rows"]),
                f"{r['n_flags']:,}",
                f"{r['output_mb']}",
                f"{r['elapsed_s']}",
                status,
            )

        console.print(t)
        n_ok  = sum(1 for r in self._all_results if r["status"] == "OK")
        n_err = sum(1 for r in self._all_results if r["status"] != "OK")
        console.print(
            Panel(
                f"[success]PIPELINE COMPLETO EM {elapsed_total:.2f}s[/]\n\n"
                f"   [dim]Cenários     : {len(self.scenarios)} (A)[/]\n"
                f"   [dim]Estratégias  : {len(self.strategies)}[/]\n"
                f"   [dim]Splits       : TREINO + TESTE[/]\n"
                f"   [dim]Arquivos OK  : {n_ok}  |  Erros: {n_err}[/]\n"
                f"   [dim]Output base  : {self.output_base}[/]",
                title="[header] RESUMO FINAL — STEP 6 [/]",
                border_style="green",
                padding=(1, 4),
            )
        )

    def _salvar_relatorio_cenario(self, results: List[Dict], dir_out: Path) -> None:
        dir_out.mkdir(parents=True, exist_ok=True)
        path = dir_out / "relatorio_imputacao.csv"
        pd.DataFrame(results).to_csv(path, index=False, encoding="utf-8-sig")
        console.print(f"   [success]✔  Relatório do cenário salvo:[/] [dim]{path}[/]\n")

    def _salvar_relatorio_consolidado(self) -> None:
        path = self.output_base / "relatorio_consolidado_todos_cenarios.csv"
        pd.DataFrame(self._all_results).to_csv(path, index=False, encoding="utf-8-sig")
        console.print(f"   [success]✔  Relatório consolidado salvo:[/] [dim]{path}[/]\n")


# ==============================================================================
# ▶️ ENTRY POINT
# ==============================================================================

if __name__ == "__main__":
    engine = MissingEngineV33(
        input_treino  = INPUT_TREINO,
        input_teste   = INPUT_TESTE,
        output_base   = OUTPUT_BASE,
        batch_size    = DEFAULT_BATCH_SIZE,
        col_chunk     = COL_CHUNK_SIZE,   # ← reduza para 200 se der OOM
        strategies    = STRATEGIES,
        scenarios     = SCENARIOS,        # apenas Cenário A
        reload_stats  = False,            # True → pula fit e reutiliza JSONs salvos
    )
    engine.run()